In [ ]:

import pandas as pd

# 定义路径
train_data_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/kidney_stone/train.csv'
test_data_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/kidney_stone/test.csv'

# 读取数据
train_df = pd.read_csv(train_data_path)
test_df = pd.read_csv(test_data_path)

train_df.head(), test_df.head()


(    id  gravity    ph  osmo  cond  urea   calc  target
 0  192    1.012  5.77   461  17.4   195   1.40       0
 1  234    1.017  5.71   704  24.5   270   3.46       0
 2    5    1.025  6.90   947  28.4   395   2.64       1
 3   45    1.008  5.98   779  17.8   418   6.99       1
 4  245    1.031  5.24   703  23.6   364  12.68       1,
     id  gravity    ph  osmo  cond  urea  calc  target
 0  358    1.025  6.03   956  27.6   473  9.39       1
 1  350    1.021  5.09   874  29.0   382  6.99       1
 2  373    1.021  5.21   725  21.4   443  3.53       1
 3  399    1.017  6.56   559  15.8   317  5.38       1
 4  369    1.011  6.79   364  15.5   159  2.64       0)

In [ ]:


# 检查缺失值
train_na = train_df.isna().sum()
test_na = test_df.isna().sum()

train_types = train_df.dtypes
test_types = test_df.dtypes

(train_na, test_na), (train_types, test_types)


((id         0
  gravity    0
  ph         0
  osmo       0
  cond       0
  urea       0
  calc       0
  target     0
  dtype: int64,
  id         0
  gravity    0
  ph         0
  osmo       0
  cond       0
  urea       0
  calc       0
  target     0
  dtype: int64),
 (id           int64
  gravity    float64
  ph         float64
  osmo         int64
  cond       float64
  urea         int64
  calc       float64
  target       int64
  dtype: object,
  id           int64
  gravity    float64
  ph         float64
  osmo         int64
  cond       float64
  urea         int64
  calc       float64
  target       int64
  dtype: object))

In [ ]:


from sklearn.model_selection import train_test_split

# 去除id列
train_df = train_df.drop(columns=['id'])
test_df = test_df.drop(columns=['id'])

# 划分特征和标签
X_train = train_df.drop(columns=['target'])
y_train = train_df['target']

# 从训练集划分出验证集 (15%)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.15, stratify=y_train, random_state=42)

# 检查划分结果
(X_train.shape, y_train.shape), (X_val.shape, y_val.shape)



(((281, 6), (281,)), ((50, 6), (50,)))

In [ ]:


from xgboost import XGBClassifier

# 创建XGBoost分类器
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)

# 在训练集上训练模型
xgb_model.fit(X_train, y_train)

# 在验证集上进行预测
y_val_pred = xgb_model.predict_proba(X_val)[:, 1]

# 计算验证集上的AUC-ROC分数
from sklearn.metrics import roc_auc_score
val_auc_roc = roc_auc_score(y_val, y_val_pred)

val_auc_roc



D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [13:47:10] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0ed59c031377d09b8-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
np.float64(0.7402597402597402)

In [ ]:


# 去除测试集中的目标列
X_test = test_df.drop(columns=['target'])
y_test = test_df['target']

# 在测试集上进行预测
y_test_pred = xgb_model.predict_proba(X_test)[:, 1]

# 计算测试集上的AUC-ROC分数
test_auc_roc = roc_auc_score(y_test, y_test_pred)

test_auc_roc



np.float64(0.7695906432748538)